# BIOPINN — data generation + training (Colab)

Generates the synthetic FDM dataset via Latin Hypercube Sampling and trains the
BIOPINN physics-informed network, then saves the processed dataset and the trained
checkpoint straight to Google Drive so they persist after this runtime is recycled.

This notebook is a **thin orchestration layer**: every piece of science/ML logic
(the FDM solver, the microenvironment model, the PINN architecture, the loss
functions, the training loop) lives in the `src/` package and is imported here
unchanged. Nothing is reimplemented in this notebook — local scripts
(`scripts/run_evaluation.py`, `scripts/run_dashboard.py`, ...) call the exact same
`src/` functions on the artifacts this notebook produces.

**Runs on Colab or locally**, automatically: it detects whether it's executing on
Google Colab (uses Drive for storage, installs deps, clones the repo if needed) or in a
normal local Jupyter kernel (uses this checkout's own `artifacts/`/`data/` folders, skips
the Drive mount, and uses your GPU if PyTorch can see one). No cells need manual editing
to switch between the two.

**Sections:** (1) setup/install &nbsp;(2) mount Drive + config &nbsp;(3) generate FDM
data &nbsp;(4) build + train the PINN &nbsp;(5) save artifacts to Drive &nbsp;(6) quick
sanity plots.

**Expected runtime:** with `QUICK_TEST = True` (default), the whole notebook runs in
a few minutes and is meant for verifying the pipeline end-to-end. Set `QUICK_TEST =
False` to generate the full 2,000-simulation dataset and train at production scale —
budget **~1–3.5 hours** for data generation (parallelized across every available CPU
core; some Latin-Hypercube-sampled small-tumor/small-nanoparticle combinations need
heavy CFL sub-stepping, see `src/fdm_solver.py`) plus the configured Adam+L-BFGS
training budget on the T4 GPU.

> **Free Colab sessions typically cap out around 12 hours (and disconnect after ~90 min idle)**, and `src/data_pipeline.py::build_dataset` doesn't checkpoint partway through -- if the runtime disconnects mid-generation, that progress is lost. For a long production run, either use Colab Pro/Pro+ for longer sessions, run data generation locally instead (more CPU cores = proportionally faster, and no session limit), or reduce `configs/default_config.yaml`'s `dataset.n_simulations` to something that reliably finishes within one session.

## 1. Setup & install

In [ ]:
import sys

# Auto-detects Colab vs. a normal local Jupyter kernel -- nothing below needs manual editing
# to switch between the two.
IN_COLAB = "google.colab" in sys.modules

# Only used on Colab (to fetch this repo's src/ package fresh into the runtime).
GITHUB_REPO_URL = "https://github.com/joshua12-5/Biopinn.git"
REPO_DIR = "/content/Biopinn"


In [ ]:
import os
from pathlib import Path


def _find_local_repo_root(start: Path) -> Path:
    """Walk upward from `start` looking for the repo root (has setup.py + src/).
    Handles this notebook being run from notebooks/, the repo root, or anywhere
    Jupyter happens to set its working directory to, as long as it's somewhere
    inside the checkout."""
    for candidate in (start, *start.parents):
        if (candidate / "setup.py").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(
        f"Could not find the BIOPINN repo root by walking up from {start}. Make sure this "
        "notebook is inside the cloned/unzipped repo (e.g. biopinn/notebooks/biopinn_train.ipynb)."
    )


if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        !git clone -q {GITHUB_REPO_URL} {REPO_DIR}
    else:
        print(f"{REPO_DIR} already exists, pulling latest instead of re-cloning.")
        !git -C {REPO_DIR} pull -q
    os.chdir(REPO_DIR)
else:
    LOCAL_REPO_DIR = _find_local_repo_root(Path.cwd())
    os.chdir(LOCAL_REPO_DIR)
    print(f"Running locally -- using this checkout at {LOCAL_REPO_DIR}")

print("cwd:", os.getcwd())


In [ ]:
# Editable install so `import src` resolves to this checkout, plus the pinned
# scientific-Python / PyTorch stack from requirements.txt. Safe to re-run locally --
# pip no-ops quickly if everything is already installed in the active environment.
!pip install -q -e .
!pip install -q -r requirements.txt


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
    if "+cpu" in torch.__version__:
        print(
            "\nNo GPU detected -- this environment has the CPU-only PyTorch build "
            f"({torch.__version__}). If this machine has an NVIDIA GPU:\n"
            "  1. Check your driver's CUDA version with `nvidia-smi` in a terminal.\n"
            "  2. Reinstall PyTorch with CUDA support in this same environment, e.g.:\n"
            "       pip uninstall -y torch\n"
            "       pip install torch --index-url https://download.pytorch.org/whl/cu121\n"
            "     (swap cu121 for the build matching your CUDA version -- see\n"
            "     https://pytorch.org/get-started/locally/ for the exact command.)\n"
            "  3. RESTART THE KERNEL and re-run this notebook from the top.\n"
            "If this machine has no NVIDIA GPU, CPU is expected -- QUICK_TEST=True still runs "
            "in a few minutes; a full-scale run (QUICK_TEST=False) will just take substantially "
            "longer than on a GPU."
        )
    else:
        print("\nNo GPU detected -- training will run on CPU (fine for QUICK_TEST=True, slow for a full-scale run).")

print("Using device:", DEVICE)


In [ ]:
import src
print("biopinn src package version:", src.__version__)

## 2. Storage location & config

**On Colab**, all data-generation and training outputs are redirected to a folder on
Google Drive (`DRIVE_OUTPUT_DIR` below) by overriding `config["paths"]`, so they survive
this runtime being recycled. `src/config.py`'s `resolve_path` joins `paths.*` onto the
repo root *unless* the path is already absolute, in which case it's used as-is -- so
pointing these at `/content/drive/...` is all that's needed.

**Running locally**, there's nothing to mount or redirect: outputs just land in this
checkout's own `artifacts/` and `data/` folders (the defaults in
`configs/default_config.yaml`), exactly where `scripts/run_evaluation.py`,
`scripts/run_dashboard.py`, etc. already expect to find them.


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Running locally -- no Drive mount needed.")


In [ ]:
# Flip to False to generate the full production dataset (2,000 sims) and train at
# full scale (20,000 Adam iters + up to 5,000 L-BFGS iters). True runs the small
# experiment_1 dev config end-to-end in a few minutes, to sanity-check the whole
# pipeline before committing to a multi-hour production run.
QUICK_TEST = True
SEED = 42

# Colab: redirect everything to a Drive folder so it survives the runtime being
# recycled. Local: leave this None -- paths.* below then keep the repo-relative
# defaults (artifacts/, data/processed/) already in configs/default_config.yaml.
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/BIOPINN_outputs" if IN_COLAB else None

# Set to True to skip section 3 below and load a dataset already generated by
# scripts/generate_dataset.py (or a previous run of this notebook) from
# paths.processed instead of regenerating it. Useful after running the standalone
# script for real multi-core speed on Windows (see section 3's markdown), or to
# resume after generation finished but training was interrupted.
DATA_ALREADY_GENERATED = False


In [ ]:
import os
from src.config import load_config, resolve_path

config = load_config("experiment_1" if QUICK_TEST else None)

if DRIVE_OUTPUT_DIR:
    # Colab: redirect every output path at the mounted Drive folder (absolute paths
    # win over the repo-relative defaults -- see the markdown note above).
    config["paths"]["processed"] = os.path.join(DRIVE_OUTPUT_DIR, "data/processed")
    config["paths"]["raw_simulations"] = os.path.join(DRIVE_OUTPUT_DIR, "data/raw_simulations")
    config["paths"]["artifacts"] = os.path.join(DRIVE_OUTPUT_DIR, "artifacts")
    config["paths"]["model_checkpoint"] = os.path.join(DRIVE_OUTPUT_DIR, "artifacts/biopinn_model.pt")
    config["paths"]["normalization_stats"] = os.path.join(DRIVE_OUTPUT_DIR, "artifacts/normalization_stats.json")
    config["paths"]["training_history"] = os.path.join(DRIVE_OUTPUT_DIR, "artifacts/training_history.json")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# Resolved once, used by every "save this artifact" cell below regardless of
# whether it's pointing at Drive (Colab) or this checkout's artifacts/ (local).
ARTIFACTS_DIR = resolve_path(config, "artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("experiment:", config["experiment_name"])
print("n_simulations:", config["dataset"]["split"]["train"] + config["dataset"]["split"]["val"] + config["dataset"]["split"]["test"])
print("device:", DEVICE)
print("outputs will be saved under:", ARTIFACTS_DIR.parent)


## 3. Generate the synthetic FDM dataset

Latin Hypercube samples the 5D parameter space (tumor radius, nanoparticle
diameter, surface concentration, decay rate, duration), solves the forward-Euler
FDM reference solution for each combination (`src/fdm_solver.py`, with the CFL
guard auto-refining the internal time step), samples data/collocation/BC/IC points
for the PINN losses, and writes normalized `train/val/test.npz` +
`normalization_stats.json` to Drive. See `src/data_pipeline.py::build_dataset`.

**Windows note:** Windows' multiprocessing can't safely parallelize this from inside
a Jupyter cell (each worker re-imports the kernel launcher as `__main__` and fails to
bootstrap, crashing with `BrokenProcessPool`), so this cell always generates serially
on Windows. For real multi-core speed, run `python scripts/generate_dataset.py` from
a terminal instead, then set `DATA_ALREADY_GENERATED = True` above and re-run this
cell to just load the result. (Colab and Linux/Mac still parallelize inline as before.)


In [ ]:
import sys
import time
from src.data_pipeline import build_dataset, load_processed_dataset

if DATA_ALREADY_GENERATED:
    print("Loading pre-generated dataset from", resolve_path(config, "processed"))
    dataset = load_processed_dataset(config)
    for split_name, tensors in dataset["splits"].items():
        print(f"  {split_name}: data_X {tensors['data_X'].shape}, collocation_X {tensors['collocation_X'].shape}")
else:
    # Windows' spawn-based multiprocessing can't parallelize this safely from inside a
    # Jupyter cell (each worker re-imports the kernel launcher as "__main__" and fails
    # to bootstrap, crashing with BrokenProcessPool). Generate serially here instead --
    # for real parallel speed on Windows, run `python scripts/generate_dataset.py` from
    # a terminal first, then flip DATA_ALREADY_GENERATED to True above and re-run this
    # cell to just load the result.
    n_jobs = 1 if sys.platform == "win32" else (os.cpu_count() or 1)
    print(f"Solving FDM simulations across {n_jobs} worker process(es)...")

    t0 = time.time()
    dataset = build_dataset(config, seed=SEED, save=True, n_jobs=n_jobs)
    elapsed = time.time() - t0

    print(f"\nDataset generation complete in {elapsed/60:.1f} min.")
    for split_name, tensors in dataset["splits"].items():
        print(f"  {split_name}: {len(dataset['sims'][split_name])} sims, "
              f"data_X {tensors['data_X'].shape}, collocation_X {tensors['collocation_X'].shape}")


## 4. Build & train the BIOPINN network

Two-phase training (`src/train.py::train`): Adam with StepLR decay, gradient
clipping, and a w_phys warmup ramp / NaN-recovery safeguard, followed by L-BFGS
fine-tuning with strong-Wolfe line search. Stops early once the validation physics
residual and data loss both stay under their configured thresholds for enough
consecutive epochs. Saves the best-validation checkpoint + normalization stats to
the Drive paths configured above.

**If training crashes with `CUDA out of memory`:** training is full-batch (every iteration sees the whole train split at once), and the physics loss's second-order autograd over millions of collocation points can exceed GPU memory at full production scale. `configs/default_config.yaml`'s `training.max_points_per_chunk` (default 1,000,000) already caps this by computing each loss term in point-count-bounded chunks -- mathematically identical gradient, bounded peak memory. If you still OOM, lower it (e.g. 500,000 or 250,000); if you have VRAM to spare and want marginally less looping overhead, raise it or remove the key entirely.

In [ ]:
from src.train import train

t0 = time.time()
result = train(config, dataset, device=DEVICE, save=True)
elapsed = time.time() - t0

print(f"\nTraining complete in {elapsed/60:.1f} min "
      f"({result['adam_epochs_run']} Adam epochs, "
      f"{result['lbfgs_closure_evaluations']} L-BFGS closure evaluations).")
print(f"Final validation data loss: {result['final_val_data']:.4e}")
print(f"Final validation physics residual: {result['final_val_phys']:.4e}")

## 5. Save artifacts

The processed dataset and the model checkpoint were already written directly to
their final location in steps 3-4 (Drive on Colab, this checkout's `artifacts/` /
`data/processed/` locally -- via the `paths.*` overrides in section 2) -- nothing
left to copy. This cell just confirms the files landed and additionally saves the
loss history + the exact config used, for reproducibility and for the sanity plots
below.

In [ ]:
import json

artifact_paths = result["artifacts"]
print("checkpoint:", artifact_paths["checkpoint_path"], "(",
      os.path.getsize(artifact_paths["checkpoint_path"]) / 1e6, "MB )")
print("normalization stats:", artifact_paths["normalization_stats_path"])

for split_name in ("train", "val", "test"):
    npz_path = os.path.join(config["paths"]["processed"], f"{split_name}.npz")
    print(split_name, "->", npz_path, "(", os.path.getsize(npz_path) / 1e6, "MB )")

In [ ]:
# Loss history + the exact resolved config, alongside the checkpoint, so a later
# local session can reproduce the run's plots without re-training.
run_record_path = ARTIFACTS_DIR / "training_run.json"
with open(run_record_path, "w", encoding="utf-8") as f:
    json.dump({
        "config": config,
        "history": result["history"],
        "adam_epochs_run": result["adam_epochs_run"],
        "lbfgs_closure_evaluations": result["lbfgs_closure_evaluations"],
        "final_val_data": result["final_val_data"],
        "final_val_phys": result["final_val_phys"],
    }, f, indent=2)
print("saved run record:", run_record_path)
# Also save just the loss history under the exact filename Phase 14's
# results-generation pipeline expects (src/results.py looks for this first,
# falling back to extracting "history" from training_run.json above).
history_path = resolve_path(config, "training_history")  # respects paths.training_history (e.g. experiment_1's "_dev" suffix)
with open(history_path, "w", encoding="utf-8") as f:
    json.dump(result["history"], f, indent=2)
print("saved training history:", history_path)


## 6. Quick sanity plots

Loss curves per component, and a predicted-vs-FDM-reference concentration profile
for one held-out validation simulation, so an obviously broken run is caught here
rather than three phases later in `scripts/run_evaluation.py`.

In [ ]:
import matplotlib.pyplot as plt

history = result["history"]
adam_n = result["adam_epochs_run"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, key in zip(axes.ravel(), ("total", "data", "phys", "bc", "neu", "ic")):
    vals = history[key]
    if any(v > 0 for v in vals):
        ax.semilogy(vals)
    else:
        ax.plot(vals)  # identically-zero series (e.g. ic, via the hard-IC transform)
    ax.axvline(adam_n, color="gray", linestyle="--", linewidth=1)
    ax.set_title(f"{key} loss")
    ax.set_xlabel("iteration")
    ax.grid(alpha=0.3)
plt.suptitle(f"BIOPINN training loss curves ({config['experiment_name']})")
plt.tight_layout()
fig_path = ARTIFACTS_DIR / "training_loss_curves.png"
plt.savefig(fig_path, dpi=150)
print("saved:", fig_path)
plt.show()

In [ ]:
import json
import numpy as np
import torch

from src.data_pipeline import PARAM_ORDER, _solve_one
from src.model import BIOPINN

# Pick one validation-split simulation and compare the trained model's prediction
# against its own FDM reference solution.
if "sims" in dataset:
    val_sim = dataset["sims"]["val"][0]
else:
    # DATA_ALREADY_GENERATED=True: load_processed_dataset doesn't keep the raw
    # (r, t, C) arrays in memory, so re-solve one val-split simulation on demand
    # from the exact parameters saved in sim_params.json -- the same approach
    # src/evaluate.py uses to reconstruct the test split's reference fields.
    processed_dir = resolve_path(config, "processed")
    with open(processed_dir / "sim_params.json", encoding="utf-8") as f:
        val_params = json.load(f)["val"][0]
    val_sim = _solve_one((val_params["sim_id"], np.array([val_params[k] for k in PARAM_ORDER]), config))

r, t, C_fdm = val_sim["r"], val_sim["t"], val_sim["C"]
stats = dataset["stats"]

def normalize_param(value, key):
    lo, hi = stats[key]["min"], stats[key]["max"]
    return 0.0 if hi <= lo else (value - lo) / (hi - lo)

param_row = [normalize_param(val_sim[k], k) for k in ("R_um", "d_NP_nm", "C0_uM", "k_d_per_hr", "t_max_hr")]

model = result["model"].to("cpu").eval()
t_snapshot = val_sim["t_max_hr"]  # final time
t_idx = np.argmin(np.abs(t - t_snapshot))

r_norm = torch.tensor(r / val_sim["R_um"], dtype=torch.float32).reshape(-1, 1)
t_norm = torch.full_like(r_norm, t[t_idx] / val_sim["t_max_hr"])
param_cols = torch.tensor(param_row, dtype=torch.float32).repeat(len(r), 1)
X = torch.cat([r_norm, t_norm, param_cols], dim=1)

with torch.no_grad():
    C_pred = (model(X).numpy().ravel()) * val_sim["C0_uM"]

plt.figure(figsize=(7, 5))
plt.plot(r, C_fdm[t_idx, :], "k-", linewidth=2, label="FDM reference")
plt.plot(r, C_pred, "r--", linewidth=2, label="PINN prediction")
plt.xlabel("radius (um)")
plt.ylabel("concentration (uM)")
plt.title(f"Validation sim {val_sim['sim_id']} at t={t[t_idx]:.1f}hr "
          f"(R={val_sim['R_um']:.0f}um, d_NP={val_sim['d_NP_nm']:.0f}nm)")
plt.legend()
plt.grid(alpha=0.3)
fig_path = ARTIFACTS_DIR / "validation_sanity_check.png"
plt.savefig(fig_path, dpi=150)
print("saved:", fig_path)
plt.show()


## Next steps

**If you ran this on Colab:** download `artifacts/biopinn_model.pt`,
`artifacts/normalization_stats.json`, and `artifacts/training_history.json` from
Drive into your local repo's `artifacts/` folder, and the `data/processed/*.npz`
files into `data/processed/`.

**If you ran this locally:** nothing to move -- steps 3-5 above already wrote
everything directly into this checkout's `artifacts/` and `data/processed/`.

Either way, from here on (CPU is enough -- nothing below here re-trains):

```bash
python scripts/run_evaluation.py    # six-metric report + H1/H2/H4 pass/fail
python scripts/run_ablation.py      # physics-informed vs. unconstrained baseline
python scripts/run_optimization.py  # optimal (d_NP*, C0*) per tumor radius
python scripts/make_figures.py      # publication figures
python scripts/run_dashboard.py     # interactive results dashboard
python scripts/generate_results.py  # manuscript figures + tables -> results/paper/
```
